In [1]:
!pip install -U transformers==4.41.2
!pip install -U datasets==2.19.1
!pip install -U accelerate==0.31.0
!pip install -U evaluate==0.4.2
!pip install -U torch==2.1.2 --index-url https://download.pytorch.org/whl/cu118


Looking in indexes: https://download.pytorch.org/whl/cu118


In [4]:
import pandas as pd
from datasets import load_dataset

# Load full dataset from HF
dataset = load_dataset("srikanthgali/ai-text-detection-pile-cleaned")

# Convert to pandas for easy subsetting
df = dataset['train'].to_pandas()

# Balance & subsample (20k human + 20k AI)
df_sampled = (
    df.groupby("generated")
      .apply(lambda x: x.sample(min(20000, len(x)), random_state=42))
      .reset_index(drop=True)
)

df_sampled.head(), df_sampled.generated.value_counts()



/tmp/ipykernel_286/990031769.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(20000, len(x)), random_state=42))


(                                                text  generated source
 0  How to Trim your Climbing Rope Sport climbers ...          0  human
 1  There  s a princess in that tower, and I  m go...          0  human
 2  Zorb. Zorb. dammit, Zorb, you gon na hand me t...          0  human
 3  We had been on the hunt for treasure for what ...          0  human
 4  The most profoundly honest and deep conversati...          0  human,
 generated
 0    20000
 1    20000
 Name: count, dtype: int64)

In [5]:
from datasets import Dataset, ClassLabel

# Convert pandas → HF dataset
dataset_small = Dataset.from_pandas(df_sampled)

# Keep only text + label
dataset_small = dataset_small.remove_columns(
    [col for col in dataset_small.column_names if col not in ["text", "generated"]]
)

# Rename
dataset_small = dataset_small.rename_column("generated", "labels")

# Convert labels → ClassLabel (0,1)
class_label = ClassLabel(names=["human", "ai"])

dataset_small = dataset_small.cast_column("labels", class_label)

# Now split with stratification (works!)
dataset_small = dataset_small.train_test_split(
    test_size=0.2,
    stratify_by_column="labels"
)

train_ds = dataset_small["train"]
val_ds = dataset_small["test"]

train_ds, val_ds


Casting the dataset:   0%|          | 0/40000 [00:00<?, ? examples/s]

(Dataset({
     features: ['text', 'labels'],
     num_rows: 32000
 }),
 Dataset({
     features: ['text', 'labels'],
     num_rows: 8000
 }))

In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds.set_format("torch")
val_ds.set_format("torch")


Map:   0%|          | 0/32000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

# Training

In [13]:
#RUN force-install compatible versions that work together 
# !pip uninstall -y transformers accelerate peft protobuf
# !pip install transformers==4.41.2
# !pip install accelerate==0.31.0
# !pip install peft==0.11.1
# !pip install protobuf==3.20.*

# Remove the incompatible versions and install compatible ones
!pip uninstall -y transformers accelerate
!pip install transformers==4.41.2
!pip install accelerate==0.31.0

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found existing installation: transformers 4.40.2
Uninstalling transformers-4.40.2:
  Successfully uninstalled transformers-4.40.2
Found existing installation: accelerate 0.30.1
Uninstalling accelerate-0.30.1:
  Successfully uninstalled accelerate-0.30.1


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.11.1 requires accelerate>=0.21.0, which is not installed.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached accelerate-0.31.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-0.31.0-py3-none-any.whl (309 kB)


In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

training_args = TrainingArguments(
    output_dir="./roberta-output",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    fp16=True,
    report_to="none",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


2025-12-02 20:37:36.720969: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764707856.753179     286 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764707856.760785     286 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [8]:
from accelerate import Accelerator

accelerator = Accelerator(mixed_precision="fp16")


In [9]:
trainer.train()


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.167600,0.190215,0.939250,0.939047
2,0.061100,0.264924,0.940125,0.939924


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


TrainOutput(global_step=1000, training_loss=0.11435744285583496, metrics={'train_runtime': 2252.2783, 'train_samples_per_second': 28.416, 'train_steps_per_second': 0.444, 'total_flos': 8419553771520000.0, 'train_loss': 0.11435744285583496, 'epoch': 2.0})

In [10]:
# === SAVE THE TRAINED MODEL CORRECTLY ===

save_dir = "./final_roberta_model"

trainer.save_model(save_dir)          # saves model weights + tokenizer + config
tokenizer.save_pretrained(save_dir)
model.config.save_pretrained(save_dir)

print("Model saved to:", save_dir)


Model saved to: ./final_roberta_model


In [11]:
!zip -r roberta_model.zip final_roberta_model


  adding: final_roberta_model/ (stored 0%)
  adding: final_roberta_model/special_tokens_map.json (deflated 52%)
  adding: final_roberta_model/merges.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 53%)
  adding: final_roberta_model/model.safetensors (deflated 8%)
  adding: final_roberta_model/tokenizer_config.json (deflated 76%)
  adding: final_roberta_model/vocab.json (deflated 59%)
  adding: final_roberta_model/tokenizer.json (deflated 72%)
  adding: final_roberta_model/config.json (deflated 51%)
  adding: final_roberta_model/training_args.bin (deflated 52%)
